<a href="https://colab.research.google.com/github/pandeynivedita7/codingal/blob/main/Voice_Activated_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import speech_recognition as sr#•	speech_recognition for speech-to-text (listening).
import pyttsx3#•	pyttsx3 for text-to-speech (speaking).
from datetime import datetime#•	datetime to respond with the current time.

# Function to make the assistant speak
def speak(text):
    engine = pyttsx3.init()# Initialize text-to-speech engine
    engine.setProperty('rate', 150)  # Set speech speed # Set speaking rate (words per minute)
    engine.say(text)# Add the text to speak
    engine.runAndWait()# Speak out loud

# Function to get audio input from the user and convert to text
def get_audio():
    r = sr.Recognizer()## Create Recognizer instance
    with sr.Microphone() as source:# Use microphone for input
        print("🎤 Speak now...")
        audio = r.listen(source)# Listen for input
        try:
            command = r.recognize_google(audio)# Convert speech to text using Google
            print(f"✅ You said: {command}")
            return command.lower()# Return in lowercase for easier matching
        except sr.UnknownValueError:
            print("❌ Could not understand.")
        except sr.RequestError as e:
            print(f"❌ API Error: {e}")
    return ""# Return empty string if there’s an error

# Function to respond based on voice commands
def respond_to_command(command):
    if "hello" in command:
        speak("Hi there! How can I help you today?")
    elif "your name" in command:
        speak("I am your Python voice assistant.")
    elif "time" in command:
        now = datetime.now().strftime("%H:%M")# Get current time
        speak(f"The time is {now}")
    elif "exit" in command or "stop" in command:
        speak("Goodbye!")
        return False# Stop the assistant
    else:
        speak("I'm not sure how to help with that.")
    return True# Continue running

# Main function to activate the assistant
def main():
    speak("Voice assistant activated. Say something!")
    while True:
        command = get_audio()
        if command and not respond_to_command(command):
            break# Exit if respond_to_command returns False

# Run the assistant
if __name__ == "__main__":
    main()
#•	“Hello” → 💬 “Hi there! How can I help you today?”
#	“What’s your name?” → 💬 “I am your Python voice assistant.”
#	“What time is it?” → 💬 “The time is 16:42”
#	“Exit” or “Stop” → 💬 “Goodbye!” (and the program ends)

In [ ]:
import sounddevice as sd
import speech_recognition as sr
import numpy as np

def get_audio():
    recognizer = sr.Recognizer()

    samplerate = 16000
    duration = 5  # seconds

    print("Listening...")

    # Record audio normally
    audio_data = sd.rec(int(samplerate * duration),
                        samplerate=samplerate,
                        channels=1,
                        dtype='int16')
    sd.wait()

    # Convert numpy → proper bytes
    audio_bytes = audio_data.tobytes()

    # Create proper AudioData object
    audio = sr.AudioData(audio_bytes, samplerate, sample_width=2)

    try:
        text = recognizer.recognize_google(audio)
        print("You said:", text)
        return text.lower()
    except sr.UnknownValueError:
        print("Could not understand speech.")
    except sr.RequestError as e:
        print("API Error:", e)

    return ""



In [ ]:
"""
Smart Command Pro — Upgraded Voice Assistant (ready-to-run)

Dependencies:
    pip install SpeechRecognition pyttsx3 pyaudio

Notes:
- On Windows, if pip install pyaudio fails, install a matching wheel from:
  https://www.lfd.uci.edu/~gohlke/pythonlibs/#pyaudio
- Test microphone input and system audio devices before running.
- This script uses offline TTS (pyttsx3) and the Google Web Speech API via
  SpeechRecognition (requires internet for recognition).

Features:
- Continuous listening loop with ambient-noise calibration
- Robust error handling and timeouts
- Command parsing and easy-to-extend command map
- Date/time utilities
- Utility commands: repeat, search web, open apps (platform-aware), tell joke
- Graceful shutdown on "exit" or "stop"
"""

import sys
import platform
import subprocess
import webbrowser
from datetime import datetime
import speech_recognition as sr
import pyttsx3
import time

# -----------------------------
# Configuration
# -----------------------------
LISTEN_TIMEOUT = 5           # seconds to wait for phrase to start
PHRASE_TIME_LIMIT = 6        # seconds max per phrase
AMBIENT_ADJUST_DURATION = 1  # seconds to sample ambient noise
SPEECH_RATE = 150
VOICE_VOLUME = 1.0           # 0.0 to 1.0

# -----------------------------
# Initialize TTS engine (global)
# -----------------------------
engine = pyttsx3.init()
engine.setProperty("rate", SPEECH_RATE)
engine.setProperty("volume", VOICE_VOLUME)
# Optionally set voice by name/index:
# voices = engine.getProperty("voices")
# engine.setProperty("voice", voices[0].id)

def speak(text, wait=True):
    """
    Speak the provided text using pyttsx3.
    If wait is False, it will queue speech and return immediately.
    """
    if not text:
        return
    engine.say(text)
    if wait:
        engine.runAndWait()
    else:
        # Non-blocking: start the loop briefly to flush queued items.
        engine.startLoop(False)

# -----------------------------
# Utilities
# -----------------------------
def get_time_text():
    now = datetime.now()
    return now.strftime("%I:%M %p")  # 12-hour format with AM/PM

def get_date_text():
    now = datetime.now()
    return now.strftime("%A, %B %d, %Y")

def tell_joke():
    jokes = [
        "Why do programmers prefer dark mode? Because light attracts bugs.",
        "Why did the programmer quit his job? Because he didn't get arrays.",
        "Why do Java developers wear glasses? Because they don't C#."
    ]
    return jokes[int(time.time()) % len(jokes)]

def open_application(name):
    """
    Try to open common apps. Extend as needed. Returns (success, message).
    """
    system = platform.system().lower()
    name = name.lower()
    try:
        if "notepad" in name or "editor" in name:
            if system == "windows":
                subprocess.Popen(["notepad"])
            elif system == "darwin":  # macOS
                subprocess.Popen(["open", "-a", "TextEdit"])
            else:  # linux
                subprocess.Popen(["xdg-open", "."])
            return True, "Opened text editor."
        if "calculator" in name:
            if system == "windows":
                subprocess.Popen(["calc"])
            elif system == "darwin":
                subprocess.Popen(["open", "-a", "Calculator"])
            else:
                # Many distros use gnome-calculator; fallback to xcalc
                try:
                    subprocess.Popen(["gnome-calculator"])
                except Exception:
                    subprocess.Popen(["xcalc"])
            return True, "Opened calculator."
        # Generic: try to open by executable name
        subprocess.Popen([name])
        return True, f"Attempted to open {name}."
    except Exception as e:
        return False, f"Could not open {name}: {e}"

# -----------------------------
# Speech recognition helpers
# -----------------------------
recognizer = sr.Recognizer()

def listen_once():
    """
    Listen once and return recognized text (lowercased).
    Returns empty string on failure.
    """
    with sr.Microphone() as source:
        # Calibrate for ambient noise (short)
        recognizer.adjust_for_ambient_noise(source, duration=AMBIENT_ADJUST_DURATION)
        print("Listening...")
        try:
            audio = recognizer.listen(source, timeout=LISTEN_TIMEOUT, phrase_time_limit=PHRASE_TIME_LIMIT)
        except sr.WaitTimeoutError:
            print("No speech detected (timeout).")
            return ""
    try:
        text = recognizer.recognize_google(audio)
        print("Heard:", text)
        return text.lower()
    except sr.UnknownValueError:
        print("Could not understand audio.")
    except sr.RequestError as e:
        print("Speech recognition API error:", e)
    except Exception as e:
        print("Unexpected recognition error:", e)
    return ""

# -----------------------------
# Command processing
# -----------------------------
def process_command(command):
    """
    Returns False when the main loop should exit; True to continue.
    """
    if not command:
        # No command — keep listening
        return True

    # Exit commands
    if any(kw in command for kw in ("exit", "stop", "goodbye", "shutdown", "quit")):
        speak("Goodbye. Shutting down.")
        return False

    # Greeting
    if "hello" in command or "hi" in command or "hey" in command:
        speak("Hello. How can I assist you?")

    # Identity
    elif "your name" in command or "who are you" in command:
        speak("I am Smart Command Pro, your voice assistant.")

    # Time and date
    elif "time" in command:
        speak(f"The time is {get_time_text()}.")

    elif "date" in command or "today" in command:
        speak(f"Today is {get_date_text()}.")

    # Day name
    elif "day" in command and "today" in command:
        speak(f"Today is {datetime.now().strftime('%A')}.")

    # Repeat after me
    elif command.startswith("repeat after me"):
        to_repeat = command.replace("repeat after me", "", 1).strip()
        if to_repeat:
            speak(to_repeat)
        else:
            speak("What should I repeat?")

    # Search web
    elif command.startswith("search for") or command.startswith("search"):
        # Extract query
        if "search for" in command:
            query = command.split("search for", 1)[1].strip()
        else:
            query = command.split("search", 1)[1].strip()
        if query:
            speak(f"Searching the web for {query}")
            url = "https://www.google.com/search?q=" + webbrowser.quote(query)
            webbrowser.open(url)
        else:
            speak("Please tell me what to search for.")

    # Tell joke
    elif "joke" in command or "make me laugh" in command:
        speak(tell_joke())

    # Open applications
    elif "open" in command:
        # e.g., "open notepad" or "open calculator"
        target = command.split("open", 1)[1].strip()
        if target:
            success, msg = open_application(target)
            speak(msg)
        else:
            speak("Which application should I open?")

    # How are you
    elif "how are you" in command:
        speak("I am running smoothly. How can I help you?")

    # Fallback -- echo and offer help
    else:
        # Provide a helpful fallback: summarize supported actions
        speak("I didn't understand that command. I can tell the time, the date, open apps, search the web, repeat text, or tell a joke. Try one of those.")
    return True

# -----------------------------
# Main loop
# -----------------------------
def main():
    # Initial greeting (spoken)
    speak("Smart Command Pro activated. Say a command when ready.")
    running = True
    while running:
        try:
            cmd = listen_once()
            running = process_command(cmd)
        except KeyboardInterrupt:
            # Graceful exit on Ctrl+C
            speak("Interrupted. Shutting down.")
            break
        except Exception as e:
            # Catch-all to prevent crash, report briefly, and continue
            print("Error in main loop:", e)
            speak("An error occurred. I am ready for the next command.")
    # Ensure engine stops
    try:
        engine.stop()
    except Exception:
        pass

if __name__ == "__main__":
    main()


In [ ]:
import sounddevice as sd
from vosk import Model, KaldiRecognizer
import pyttsx3
import json
import datetime
import webbrowser
import queue
import sys

class VoiceAssistant:
    def __init__(self, model_path="model"):
        # Initialize Vosk model and recognizer
        try:
            self.model = Model(model_path)
            self.recognizer = KaldiRecognizer(self.model, 16000)
        except:
            print("ERROR: Please download Vosk model first!")
            print("Download from: https://alphacephei.com/vosk/models")
            print("Extract to a folder named 'model' in the same directory")
            sys.exit(1)

        # Initialize audio queue
        self.audio_queue = queue.Queue()

        # Initialize TTS engine
        self.tts_engine = pyttsx3.init()
        self.setup_voice()

        self.is_listening = True

    def setup_voice(self):
        """Configure voice properties"""
        voices = self.tts_engine.getProperty('voices')
        self.tts_engine.setProperty('voice', voices[0].id)
        self.tts_engine.setProperty('rate', 180)
        self.tts_engine.setProperty('volume', 0.9)

    def speak(self, text):
        """Convert text to speech"""
        print(f"Assistant: {text}")
        self.tts_engine.say(text)
        self.tts_engine.runAndWait()

    def callback(self, indata, frames, time, status):
        """Callback function to capture audio data"""
        if status:
            print(status)
        self.audio_queue.put(bytes(indata))

    def listen(self):
        """Listen and recognize speech using Vosk"""
        print("Listening... (Speak now)")

        with sd.RawInputStream(samplerate=16000, blocksize=8000, dtype='int16',
                               channels=1, callback=self.callback):
            while self.is_listening:
                data = self.audio_queue.get()
                if self.recognizer.AcceptWaveform(data):
                    result = json.loads(self.recognizer.Result())
                    text = result.get('text', '')
                    if text:
                        print(f"You said: {text}")
                        return text
        return ""

    def process_query(self, query):
        """Function to process user input and respond"""
        query = query.lower()

        # Exit commands
        if any(word in query for word in ['exit', 'quit', 'bye', 'goodbye']):
            self.speak("Goodbye! Have a great day!")
            return False

        # Time
        if "time" in query:
            now = datetime.datetime.now().strftime("%I:%M %p")
            response = f"The current time is {now}"
            self.speak(response)

        # Date
        elif "date" in query or "today" in query:
            now = datetime.datetime.now().strftime("%B %d, %Y")
            response = f"Today's date is {now}"
            self.speak(response)

        # Day
        elif "day" in query:
            now = datetime.datetime.now().strftime("%A")
            response = f"Today is {now}"
            self.speak(response)

        # Search
        elif "search" in query or "google" in query:
            search_term = query.replace("search", "").replace("google", "").strip()
            if search_term:
                url = f"https://www.google.com/search?q={search_term}"
                webbrowser.open(url)
                self.speak(f"Searching for {search_term}")
            else:
                self.speak("What do you want me to search for?")

        # Open website
        elif "open youtube" in query:
            webbrowser.open("https://www.youtube.com")
            self.speak("Opening YouTube")

        elif "open google" in query:
            webbrowser.open("https://www.google.com")
            self.speak("Opening Google")

        elif "open gmail" in query:
            webbrowser.open("https://mail.google.com")
            self.speak("Opening Gmail")

        # Greetings
        elif any(word in query for word in ['hello', 'hi', 'hey']):
            self.speak("Hello! How can I help you today?")

        # How are you
        elif "how are you" in query:
            self.speak("I'm doing great! Thank you for asking. How can I assist you?")

        # Name
        elif "your name" in query or "who are you" in query:
            self.speak("I am your voice assistant powered by Vosk and pyttsx3!")

        # Help
        elif "help" in query:
            self.speak("I can tell you the time, date, search the web, open websites like YouTube and Google, and answer basic questions. Just ask me!")

        # Unknown command
        else:
            if query.strip():  # Only respond if there was actual input
                self.speak("I'm not sure how to help with that. Try saying 'help' to see what I can do.")

        return True

    def run(self):
        """Main function to run the assistant"""
        self.speak("Hello! I am your voice assistant. How can I help you?")
        print("\n" + "="*50)
        print("Voice Assistant is running...")
        print("="*50)
        print("\nCommands you can try:")
        print("- 'What time is it?'")
        print("- 'What's the date?'")
        print("- 'Search Python tutorials'")
        print("- 'Open YouTube'")
        print("- 'Help'")
        print("- 'Exit' or 'Bye'")
        print("="*50 + "\n")

        try:
            while True:
                query = self.listen()
                if query:
                    if not self.process_query(query):
                        break
        except KeyboardInterrupt:
            print("\n\nStopping assistant...")
            self.speak("Goodbye!")
        except Exception as e:
            print(f"Error: {e}")
            self.speak("Sorry, I encountered an error.")

# Run the assistant
if __name__ == "__main__":
    print("\n" + "="*50)
    print("VOICE ASSISTANT with VOSK")
    print("="*50)
    print("\nBefore running, make sure you have:")
    print("1. Downloaded a Vosk model from https://alphacephei.com/vosk/models")
    print("2. Extracted it to a folder named 'model' in this directory")
    print("3. Installed required packages:")
    print("   pip install vosk sounddevice pyttsx3")
    print("="*50 + "\n")

    input("Press Enter to start the assistant...")

    assistant = VoiceAssistant()
    assistant.run()

In [ ]:
import sounddevice as sd
from vosk import Model, KaldiRecognizer
import pyttsx3
import json
import datetime
import webbrowser
import queue
import sys
#1. Go to: https://alphacephei.com/vosk/models
#2. Download a model (recommended: **vosk-model-small-en-us-0.15** for English - ~40MB)
#Extract the downloaded folder
#4. Rename it to **"model"**
#5. Place it in the same directory as your Python script
class VoiceAssistant:
    def __init__(self, model_path="model"):
        # Initialize Vosk model and recognizer
        try:
            self.model = Model(model_path)
            self.recognizer = KaldiRecognizer(self.model, 16000)
        except:
            print("ERROR: Please download Vosk model first!")
            print("Download from: https://alphacephei.com/vosk/models")
            print("Extract to a folder named 'model' in the same directory")
            sys.exit(1)

        # Initialize audio queue
        self.audio_queue = queue.Queue()

        # Initialize TTS engine
        self.tts_engine = pyttsx3.init()
        self.setup_voice()

        self.is_listening = True

    def setup_voice(self):
        """Configure voice properties"""
        voices = self.tts_engine.getProperty('voices')
        self.tts_engine.setProperty('voice', voices[0].id)
        self.tts_engine.setProperty('rate', 180)
        self.tts_engine.setProperty('volume', 0.9)

    def speak(self, text):
        """Convert text to speech"""
        print(f"Assistant: {text}")
        self.tts_engine.say(text)
        self.tts_engine.runAndWait()

    def callback(self, indata, frames, time, status):
        """Callback function to capture audio data"""
        if status:
            print(status)
        self.audio_queue.put(bytes(indata))

    def listen(self):
        """Listen and recognize speech using Vosk"""
        print("Listening... (Speak now)")

        with sd.RawInputStream(samplerate=16000, blocksize=8000, dtype='int16',
                               channels=1, callback=self.callback):
            while self.is_listening:
                data = self.audio_queue.get()
                if self.recognizer.AcceptWaveform(data):
                    result = json.loads(self.recognizer.Result())
                    text = result.get('text', '')
                    if text:
                        print(f"You said: {text}")
                        return text
        return ""

    def process_query(self, query):
        """Function to process user input and respond"""
        query = query.lower()

        # Exit commands
        if any(word in query for word in ['exit', 'quit', 'bye', 'goodbye']):
            self.speak("Goodbye! Have a great day!")
            return False

        # Time
        if "time" in query:
            now = datetime.datetime.now().strftime("%I:%M %p")
            response = f"The current time is {now}"
            self.speak(response)

        # Date
        elif "date" in query or "today" in query:
            now = datetime.datetime.now().strftime("%B %d, %Y")
            response = f"Today's date is {now}"
            self.speak(response)

        # Day
        elif "day" in query:
            now = datetime.datetime.now().strftime("%A")
            response = f"Today is {now}"
            self.speak(response)

        # Search
        elif "search" in query or "google" in query:
            search_term = query.replace("search", "").replace("google", "").strip()
            if search_term:
                url = f"https://www.google.com/search?q={search_term}"
                webbrowser.open(url)
                self.speak(f"Searching for {search_term}")
            else:
                self.speak("What do you want me to search for?")

        # Open website
        elif "open youtube" in query:
            webbrowser.open("https://www.youtube.com")
            self.speak("Opening YouTube")

        elif "open google" in query:
            webbrowser.open("https://www.google.com")
            self.speak("Opening Google")

        elif "open gmail" in query:
            webbrowser.open("https://mail.google.com")
            self.speak("Opening Gmail")

        # Greetings
        elif any(word in query for word in ['hello', 'hi', 'hey']):
            self.speak("Hello! How can I help you today?")

        # How are you
        elif "how are you" in query:
            self.speak("I'm doing great! Thank you for asking. How can I assist you?")

        # Name
        elif "your name" in query or "who are you" in query:
            self.speak("I am your voice assistant powered by Vosk and pyttsx3!")

        # Help
        elif "help" in query:
            self.speak("I can tell you the time, date, search the web, open websites like YouTube and Google, and answer basic questions. Just ask me!")

        # Unknown command
        else:
            if query.strip():  # Only respond if there was actual input
                self.speak("I'm not sure how to help with that. Try saying 'help' to see what I can do.")

        return True

    def run(self):
        """Main function to run the assistant"""
        self.speak("Hello! I am your voice assistant. How can I help you?")
        print("\n" + "="*50)
        print("Voice Assistant is running...")
        print("="*50)
        print("\nCommands you can try:")
        print("- 'What time is it?'")
        print("- 'What's the date?'")
        print("- 'Search Python tutorials'")
        print("- 'Open YouTube'")
        print("- 'Help'")
        print("- 'Exit' or 'Bye'")
        print("="*50 + "\n")

        try:
            while True:
                query = self.listen()
                if query:
                    if not self.process_query(query):
                        break
        except KeyboardInterrupt:
            print("\n\nStopping assistant...")
            self.speak("Goodbye!")
        except Exception as e:
            print(f"Error: {e}")
            self.speak("Sorry, I encountered an error.")

# Run the assistant
if __name__ == "__main__":
    print("\n" + "="*50)
    print("VOICE ASSISTANT with VOSK")
    print("="*50)
    print("\nBefore running, make sure you have:")
    print("1. Downloaded a Vosk model from https://alphacephei.com/vosk/models")
    print("2. Extracted it to a folder named 'model' in this directory")
    print("3. Installed required packages:")
    print("   pip install vosk sounddevice pyttsx3")
    print("="*50 + "\n")

    input("Press Enter to start the assistant...")

    assistant = VoiceAssistant()
    assistant.run()